# EcoRewind 2.0 — Phase 3: deep models (Colab **T4 GPU, High-RAM**)

Runtime → Change runtime type → **T4 GPU**, and turn on **High-RAM**. Don't pick an A100.
Order (MASTER_PROMPT §8): 1. smoke test → 2. timing run (EcoTransformer v2, 1 fold, 1 seed, Ida Track A) → **stop and send me the log**.
Everything resumes from Drive if Colab disconnects.

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')
import os
TOKEN = userdata.get('GITHUB_TOKEN')
REPO = '/content/ecorewind2'
if os.path.isdir(REPO):
    !git -C {REPO} pull -q
else:
    !git clone -q https://{TOKEN}@github.com/DAISHINKAN7/EcoRewind2.git {REPO}
%cd {REPO}
!git log --oneline -1
!pip -q install -r requirements.txt
!nvidia-smi --query-gpu=name,memory.total --format=csv
D = '/content/drive/MyDrive/EcoRewind2'

### 1. Tests (includes a CPU smoke test of every deep model through the harness)

In [ ]:
!python -m pytest -q tests

### 2. Cubes from Drive (~5 min)

In [ ]:
!python scripts/cubes_cache.py --drive {D}/cubes --local /content/cubes --exports-drive /content/drive/MyDrive/EcoRewind2_exports

In [ ]:
!git -C /content/ecorewind2 pull -q

### 3. Timing run: EcoTransformer v2, test fold 0, seed 0, Ida Track A (event + 2018 placebo)

In [ ]:
import time; t0 = time.time()
!python scripts/run_baselines.py --cubes /content/cubes --phase1 {D}/phase1 --out {D}/runs --run-id phase3_timing --site mississippi --track A --folds 0 --models ecotransformer_s0 --table-cache /content/tables 2>&1 | tee /content/timing.log | grep -v "^|"
print(f'wall time {(time.time()-t0)/60:.1f} min')
!cp /content/timing.log {D}/runs/phase3_timing/

### 4. Main run (MASTER_PROMPT §8 step 3): 3 core models × 5 folds × 2 seeds, Irma Track B then Ida Track A
~40 h of T4 in total. **Resumable**: every trained (model, cutoff, fold) is cached on Drive, so run this cell once per
Colab session (a session lasts a few hours), reconnect, re-run cells 0–2 and this cell, and it continues where it stopped.

In [ ]:
RUN = 'phase3_core_v1'
MODELS = 'ecotransformer_s0 simvp_s0 convlstm_s0 ecotransformer_s1 simvp_s1 convlstm_s1'
!python -u scripts/run_baselines.py --cubes /content/cubes --phase1 {D}/phase1 --out {D}/runs --run-id {RUN} --site everglades --track B --models {MODELS} --table-cache /content/tables 2>&1 | tee -a /content/core.log
!python -u scripts/run_baselines.py --cubes /content/cubes --phase1 {D}/phase1 --out {D}/runs --run-id {RUN} --site mississippi --track A --models {MODELS} --table-cache /content/tables 2>&1 | tee -a /content/core.log
!cp /content/core.log {D}/runs/{RUN}/core_$(date +%s).log

When both commands finish, send me `runs/phase3_core_v1/summary.md` and the CCU balance. **Runtime → Disconnect and delete runtime** after each session.

# `Phase 4`

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')
import os
REPO = '/content/ecorewind2'
if os.path.isdir(REPO):
    !git -C {REPO} pull -q
else:
    !git clone -q https://{userdata.get('GITHUB_TOKEN')}@github.com/DAISHINKAN7/EcoRewind2.git {REPO}
%cd {REPO}
!pip -q install -r requirements.txt
D = '/content/drive/MyDrive/EcoRewind2'
!python scripts/placebo_verdict.py --runs {D}/runs/phase2_baselines_v3 {D}/runs/phase3_core_v1 --phase1 {D}/phase1 --out {D}/reports

In [ ]:
!git -C /content/ecorewind2 pull -q
!python scripts/placebo_verdict.py --runs {D}/runs/phase2_baselines_v3 {D}/runs/phase3_core_v1 --phase1 {D}/phase1 --out {D}/reports

# Phase 5.1

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')
import os
REPO = '/content/ecorewind2'
if os.path.isdir(REPO):
    !git -C {REPO} pull -q
else:
    !git clone -q https://{userdata.get('GITHUB_TOKEN')}@github.com/DAISHINKAN7/EcoRewind2.git {REPO}
%cd {REPO}
!pip -q install -r requirements.txt
D = '/content/drive/MyDrive/EcoRewind2'
!python scripts/attribution_spread.py --runs {D}/runs/phase2_baselines_v3 {D}/runs/phase3_core_v1 --verdict {D}/reports --out {D}/reports